# Baseline Loss Reference

This notebook compares three next-character baselines:

1. **Random predictor** (no trainable parameters)
2. **Linear predictor** (single linear layer)
3. **MLP predictor** (two layers + GeLU)

The linear and MLP baselines are trained for a fixed number of steps, and their loss curves are plotted.

In [ ]:
import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

from pocketchat.data import CharCorpus

seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device={device}")

In [ ]:
@dataclass
class BaselineConfig:
    text_path: str = "data/long_doc.txt"
    max_chars: int = 0
    vocab_size: int = 256
    seq_len: int = 64
    stride: int = 1
    batch_size: int = 64
    val_split: float = 0.1
    linear_steps: int = 400
    mlp_steps: int = 400
    learning_rate: float = 3e-4
    hidden_dim: int = 512
    log_every: int = 50

cfg = BaselineConfig()
cfg

In [ ]:
class FixedNextCharDataset(Dataset):
    """
    Input: [seq_len]
    Target: scalar ID for the next character after the window.
    """

    def __init__(self, token_ids: torch.Tensor, seq_len: int, stride: int = 1):
        if token_ids.ndim != 1:
            raise ValueError(f"token_ids must have shape [T], got {tuple(token_ids.shape)}")
        if seq_len <= 0:
            raise ValueError("seq_len must be > 0")
        if stride <= 0:
            raise ValueError("stride must be > 0")
        if token_ids.numel() <= seq_len:
            raise ValueError("token_ids must have at least seq_len + 1 elements")

        self.token_ids = token_ids.long()
        self.seq_len = seq_len
        self.stride = stride
        self.num_windows = ((token_ids.numel() - seq_len - 1) // stride) + 1

    def __len__(self):
        return self.num_windows

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.num_windows:
            raise IndexError(idx)
        start = idx * self.stride
        end = start + self.seq_len
        x = self.token_ids[start:end]
        y = self.token_ids[end]
        return x, y


corpus = CharCorpus.from_file(
    cfg.text_path,
    max_chars=cfg.max_chars,
    vocab_size=cfg.vocab_size,
)

dataset = FixedNextCharDataset(corpus.data, seq_len=cfg.seq_len, stride=cfg.stride)
val_size = max(1, int(len(dataset) * cfg.val_split))
train_size = len(dataset) - val_size
if train_size <= 0:
    raise ValueError("Not enough samples for train/validation split.")

split_generator = torch.Generator().manual_seed(seed)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=split_generator)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

print(
    f"tokens={corpus.data.numel()} vocab={corpus.vocab_size} seq_len={cfg.seq_len} \
train_windows={len(train_dataset)} val_windows={len(val_dataset)}"
)

In [ ]:
def flatten_one_hot(input_ids: torch.Tensor, vocab_size: int) -> torch.Tensor:
    one_hot = F.one_hot(input_ids, num_classes=vocab_size).to(torch.float32)
    return one_hot.reshape(input_ids.shape[0], -1)


class RandomNextCharPredictor(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.vocab_size = vocab_size

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size = input_ids.shape[0]
        return torch.randn(batch_size, self.vocab_size, device=input_ids.device)


class LinearNextCharPredictor(nn.Module):
    def __init__(self, seq_len: int, vocab_size: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.linear = nn.Linear(seq_len * vocab_size, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = flatten_one_hot(input_ids, self.vocab_size)
        return self.linear(x)


class MLPNextCharPredictor(nn.Module):
    def __init__(self, seq_len: int, vocab_size: int, hidden_dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.fc1 = nn.Linear(seq_len * vocab_size, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = flatten_one_hot(input_ids, self.vocab_size)
        x = self.act(self.fc1(x))
        return self.fc2(x)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
@torch.no_grad()
def evaluate_loss(model: nn.Module, dataloader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        total_loss += float(loss.item())
        total_batches += 1
    return total_loss / max(total_batches, 1)


def train_baseline(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    steps: int,
    lr: float,
    log_every: int = 50,
):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    history = {"step": [], "train_loss": [], "val_loss": []}
    model.train()
    train_iter = iter(train_loader)
    running_loss = 0.0
    running_count = 0

    for step in range(1, steps + 1):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item())
        running_count += 1

        should_log = (step % log_every == 0) or (step == 1) or (step == steps)
        if should_log:
            train_loss = running_loss / max(running_count, 1)
            val_loss = evaluate_loss(model, val_loader)
            history["step"].append(step)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            print(f"step={step:4d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
            running_loss = 0.0
            running_count = 0

    final_val_loss = evaluate_loss(model, val_loader)
    return final_val_loss, history

In [ ]:
results = []
histories = {}

# 1) Random baseline (no training)
random_model = RandomNextCharPredictor(vocab_size=corpus.vocab_size).to(device)
random_val_loss = evaluate_loss(random_model, val_loader)
results.append(("Random (no params)", count_params(random_model), random_val_loss))
print(f"Random baseline val_loss={random_val_loss:.4f} (uniform reference ~ ln(V)={math.log(corpus.vocab_size):.4f})")

# 2) Linear baseline (single layer)
linear_model = LinearNextCharPredictor(seq_len=cfg.seq_len, vocab_size=corpus.vocab_size)
linear_val_loss, linear_history = train_baseline(
    linear_model,
    train_loader=train_loader,
    val_loader=val_loader,
    steps=cfg.linear_steps,
    lr=cfg.learning_rate,
    log_every=cfg.log_every,
)
results.append(("Linear (1 layer)", count_params(linear_model), linear_val_loss))
histories["Linear (1 layer)"] = linear_history

# 3) MLP baseline (2 layers + GeLU)
mlp_model = MLPNextCharPredictor(
    seq_len=cfg.seq_len,
    vocab_size=corpus.vocab_size,
    hidden_dim=cfg.hidden_dim,
)
mlp_val_loss, mlp_history = train_baseline(
    mlp_model,
    train_loader=train_loader,
    val_loader=val_loader,
    steps=cfg.mlp_steps,
    lr=cfg.learning_rate,
    log_every=cfg.log_every,
)
results.append(("MLP (2 layers + GeLU)", count_params(mlp_model), mlp_val_loss))
histories["MLP (2 layers + GeLU)"] = mlp_history

In [ ]:
plt.figure(figsize=(10, 5))

for name, history in histories.items():
    plt.plot(history["step"], history["train_loss"], marker="o", label=f"{name} - train")
    plt.plot(history["step"], history["val_loss"], marker="s", linestyle="--", label=f"{name} - val")

plt.axhline(random_val_loss, color="gray", linestyle=":", label="Random baseline val")
plt.title("Baseline Loss Curves")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Baseline comparison (lower is better) ===")
print(f"{'Model':30s} {'Params':>12s} {'Val Loss':>12s}")
print('-' * 58)
for name, nparams, val_loss in results:
    print(f"{name:30s} {nparams:12,d} {val_loss:12.4f}")